# Macro Event Study: CPI → Cross-Asset Market Reaction

This notebook demonstrates an end-to-end macro event study using the
Macro Transmission Engine.

Goal:
- Quantify the macro surprise
- Measure multi-horizon market reactions
- Identify cross-asset transmission
- Provide finance-first interpretation

This mirrors how a macro or research desk would analyze a CPI release.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
sys.path.append(str(project_root))

import pandas as pd

from utils.data_loader import DataLoader
from engines.surprise_engine import MacroSurpriseEngine
from engines.reaction_engine import MarketReactionEngine
from engines.transmission_engine import TransmissionEngine


In [ ]:
loader = DataLoader(project_root)

# Load settings
event_windows = loader.get_event_windows()
pre_event_window = loader.get_pre_event_window()

# Initialize engines
surprise_engine = MacroSurpriseEngine(
    surprise_vol_window=loader.settings["surprise_vol_window"]
)

reaction_engine = MarketReactionEngine(
    event_windows=event_windows,
    pre_event_window=pre_event_window
)

transmission_engine = TransmissionEngine(
    event_windows=event_windows
)


In [ ]:
# Load CPI macro data
cpi_df = loader.load_macro_file("cpi.csv")

# Build normalized surprise series
cpi_surprise_df = surprise_engine.build_surprise_series(
    project_root / "data" / "macro" / "cpi.csv"
)

cpi_surprise_df.tail()


In [ ]:
# Select most recent CPI event
event = cpi_surprise_df.iloc[-1]

event_date = event["date"]
event_time = pd.Timestamp(
    f"{event_date.date()} 08:30",
    tz="US/Eastern"
)

event


In [ ]:
assets = loader.load_all_assets()

start = (event_time - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
end = (event_time + pd.Timedelta(days=2)).strftime("%Y-%m-%d")

market_data = {}

for group, asset in assets.items():
    if isinstance(asset, list):
        for item in asset:
            df = loader.fetch_market_data(
                ticker=item["ticker"],
                start=start,
                end=end
            )
            market_data[item["name"]] = {
                "data": df,
                "type": item["type"]
            }
    else:
        df = loader.fetch_market_data(
            ticker=asset["ticker"],
            start=start,
            end=end
        )
        market_data[asset["name"]] = {
            "data": df,
            "type": asset["type"]
        }


In [ ]:
reaction_results = {}

for asset_name, payload in market_data.items():
    reactions = reaction_engine.compute_reactions(
        market_df=payload["data"],
        event_time=event_time,
        asset_type=payload["type"]
    )
    reaction_results[asset_name] = reactions

reaction_results


In [ ]:
transmission_summary = transmission_engine.summarize_event(
    reaction_results
)

transmission_summary["event_matrix"]



In [ ]:
cpi_path = project_root / "data" / "macro" / "cpi.csv"

surprise_df = surprise_engine.build_surprise_series(cpi_path)

surprise_df.tail()

In [ ]:
historical_results = []

assets = loader.load_all_assets()

for _, event in surprise_df.iterrows():

    event_time = pd.Timestamp(
        f"{event['date'].date()} 08:30",
        tz="US/Eastern"
    )

    start = (event_time - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    end = (event_time + pd.Timedelta(days=10)).strftime("%Y-%m-%d")

    reaction_results = {}

    for group, asset in assets.items():

        if isinstance(asset, list):
            iterable = asset
        else:
            iterable = [asset]

        for item in iterable:

            df = loader.fetch_market_data(
                item["ticker"],
                start,
                end
            )

            reactions = reaction_engine.compute_reactions(
                df,
                event_time,
                item["type"]
            )

            for window, value in reactions.items():

                historical_results.append({
                    "date": event["date"],
                    "asset": item["name"],
                    "window": window,
                    "reaction": value,
                    "surprise_z": event["surprise_z"]
                })

historical_df = pd.DataFrame(historical_results)

historical_df.head()

In [ ]:
avg_reaction = (
    historical_df[historical_df["window"] == "1440m"]
    .groupby("asset")["reaction"]
    .mean()
    .sort_values()
)

avg_reaction

In [ ]:
import statsmodels.api as sm

df_reg = historical_df[
    (historical_df["window"] == "1440m") &
    (~historical_df["surprise_z"].isna())
]

for asset in df_reg["asset"].unique():

    sub = df_reg[df_reg["asset"] == asset]

    X = sm.add_constant(sub["surprise_z"])
    y = sub["reaction"]

    model = sm.OLS(y, X).fit()

    print("\n", asset)
    print(model.params)